# Lesson 18 (follow-on): Receipts That Prove a *Human* Authorized the Action

Lesson 18 showed how an **agent** signs a receipt proving *it* took an action and that the record wasn't tampered with. This follow-on adds the other half: a receipt that proves a **named human approved the *exact* action** before it ran.

These are different properties:

| Receipt | Proves |
|---|---|
| Agent action receipt (Lesson 18) | the agent did what it claims, untampered |
| **Human approval receipt (this lesson)** | **a specific person said "yes" to *this exact action*** |

They are two independent signatures over partly-overlapping fields, joined by a content-derived reference — so neither the agent, the operator, nor the human can forge the other's half.

Same primitives as Lesson 18 — Ed25519 (`pynacl`) + RFC 8785 / JCS canonicalization (`jcs`) — and everything verifies **offline**, with no service to trust.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
import jcs
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import BadSignatureError, CryptoError

def b64u(b):      return base64.urlsafe_b64encode(b).rstrip(b"=").decode()
def b64u_dec(s):  return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))
def canon(obj):   return jcs.canonicalize(obj)            # RFC 8785 -> bytes
def sha256_hex(b): return hashlib.sha256(b).hexdigest()

## The exact action

The unit of approval is the **canonical action object** — not a vague label like "approve refund," but the precise, fully-specified action. Signing over the whole object (and deriving a digest from it) is what lets us later prove the human approved *this* and nothing else.

In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_version": "refunds-v3",
}
print("action digest:", sha256_hex(canon(action)))

action digest: e68137d4f7788a7df1e64c5fe25a8e61ddb02c05a8a4f80d0cf0a5c642199138


## The human approval receipt

In production the approver signs with a **WebAuthn platform authenticator** — a passkey, Face ID, or a security key. The signature is produced on the person's own device and is verifiable against the credential's published public key (and, for an auditor, its attestation chain). That hardware origin is what makes "a human said yes" *true* rather than a self-signed claim the operator could mint.

Here we stand in an Ed25519 key for the authenticator so the notebook runs offline — the sign/verify shape is identical.

> **The one rule that matters:** a verifier checks the receipt against the approver's **published / pinned** public key — *never* the key carried inside the receipt. Trusting the embedded key is the classic mistake; an attacker simply embeds their own.

In [3]:
approver_sk = SigningKey.generate()
APPROVER_PUB = b64u(approver_sk.verify_key.encode())   # published out of band; pinned by the verifier

def human_approval(action, approver_id, approved_at_ms, sk=approver_sk):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # storing the live reference would let a later in-place mutation of `action`
    # silently change the signed payload and break verification confusingly.
    approved_action = copy.deepcopy(action)   # snapshot ONCE, then digest that snapshot
    payload = {
        "typ": "human-approval",
        "approver_id": approver_id,
        "action": approved_action,                        # the FULL canonical action, not just a digest
        # digest the snapshot, not the caller's `action`, so the two can never diverge.
        "action_digest": sha256_hex(canon(approved_action)),
        "approved_at_ms": approved_at_ms,
    }
    # approver_pub is for display only — verifiers MUST use the pinned key below.
    return {"payload": payload,
            "sig": b64u(sk.sign(canon(payload)).signature),
            "approver_pub": b64u(sk.verify_key.encode())}

def verify_human_approval(receipt, trusted_pub, action_being_executed):
    # Fail closed on ANY attacker-shaped input: a malformed receipt is an INVALID
    # receipt, never a crash. Missing fields / bad base64 must return a clean reason.
    try:
        payload, sig = receipt["payload"], receipt["sig"]
        signed_bytes, sig_bytes = canon(payload), b64u_dec(sig)
    except (KeyError, TypeError, ValueError, base64.binascii.Error):
        return (False, "receipt malformed (missing or unreadable fields)")
    # payload must be an object before we read fields off it — a list/str/number
    # payload would otherwise raise AttributeError on the .get() calls below.
    if not isinstance(payload, dict):
        return (False, "receipt malformed (payload is not an object)")
    if payload.get("typ") != "human-approval":
        return (False, "wrong receipt type")
    # Catch CryptoError (the base of BadSignatureError) so a forged signature AND a
    # wrong-length one both refuse cleanly instead of escaping as an uncaught ValueError.
    try:
        VerifyKey(b64u_dec(trusted_pub)).verify(signed_bytes, sig_bytes)
    except CryptoError:
        return (False, "signature invalid (forged or tampered)")
    if payload.get("action") != action_being_executed:
        return (False, "approval is not for the action being executed (confused deputy)")
    if payload.get("action_digest") != sha256_hex(canon(action_being_executed)):
        return (False, "action digest mismatch")
    return (True, f"approved by {payload['approver_id']}")

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", 1_750_000_000_000)
print(verify_human_approval(approval, APPROVER_PUB, action))

(True, 'approved by alice@ops (WebAuthn)')


## Joining the two receipts

The agent's action receipt (from Lesson 18) references the human approval by a **content-derived id** — `parent_approval_ref` = a hash of the approval's canonical **payload** bytes (the signed content, excluding the signature and display-only fields). Because it's derived from the bytes both sides recompute, neither side can forge the join.

We also enforce **one-time consumption**: an approval authorizes a single execution. Teaching replay-refusal matters as much as teaching signature-checking.

In [5]:
agent_sk  = SigningKey.generate()
AGENT_PUB = b64u(agent_sk.verify_key.encode())   # published out of band; pinned by the verifier

def approval_ref(receipt):                       # content-derived id of the approval
    return "ha_" + sha256_hex(canon(receipt["payload"]))   # full SHA-256 digest (no truncation)

def agent_receipt(action, approval, executed_at_ms, sk=agent_sk):
    payload = {
        "typ": "agent-action", "agent_id": "agent:refunds-bot",
        "action": copy.deepcopy(action),        # snapshot, same reason as the approval
        "parent_approval_ref": approval_ref(approval),
        "executed_at_ms": executed_at_ms,
    }
    # agent_pub is for display only — execute() uses the pinned AGENT_PUB, never this field.
    return {"payload": payload, "sig": b64u(sk.sign(canon(payload)).signature), "agent_pub": AGENT_PUB}

_consumed = set()
def execute(action, approval, agent_rcpt):
    ok, why = verify_human_approval(approval, APPROVER_PUB, action)   # pinned key, not the receipt's
    if not ok: return (False, f"human approval: {why}")
    # Fail closed on ANY structurally invalid agent receipt: malformed input is a
    # refusal, not a crash (the agent receipt is attacker-controllable too).
    try:
        a_payload, a_sig = agent_rcpt["payload"], agent_rcpt["sig"]
        a_signed, a_sig_bytes = canon(a_payload), b64u_dec(a_sig)
    except (KeyError, TypeError, ValueError, base64.binascii.Error):
        return (False, "agent receipt malformed")
    # payload must be an object before we read fields off it — a non-dict payload
    # would otherwise raise AttributeError on the .get() calls below.
    if not isinstance(a_payload, dict):
        return (False, "agent receipt malformed")
    if a_payload.get("typ") != "agent-action":
        return (False, "agent receipt malformed")
    # Verify against the PINNED agent key (never the `agent_pub` carried inside the
    # receipt) — otherwise an attacker just mints a receipt with their own key.
    # Catch CryptoError (base of BadSignatureError) so a forged OR wrong-length
    # signature both refuse cleanly instead of escaping as an uncaught ValueError.
    try:
        VerifyKey(b64u_dec(AGENT_PUB)).verify(a_signed, a_sig_bytes)
    except CryptoError:
        return (False, "agent receipt signature invalid")
    # Bind the receipt to THIS action and THIS approval, or a self-minted receipt
    # for a different action would still pass the signature check above.
    if a_payload.get("action") != action:
        return (False, "agent receipt is for a different action than the one being executed")
    if a_payload.get("parent_approval_ref") != approval_ref(approval):
        return (False, "agent receipt is not bound to this approval")
    ref = approval_ref(approval)
    if ref in _consumed: return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, "executed")

receipt = agent_receipt(action, approval, 1_750_000_000_500)
print(execute(action, approval, receipt))

(True, 'executed')


## What the binding catches

The whole point is that the approval is welded to the exact action, and that *both* signatures are checked against **pinned** keys. Each of these is refused — run and read the reason:

1. **Tampering** — the amount is changed after approval.
2. **Confused deputy** — a real approval for action A is replayed onto action B.
3. **Replay** — the same approval is used for a second execution.
4. **Forgery (human side)** — an attacker signs an *approval* with their own key.
5. **Self-minted agent receipt** — an attacker signs an *agent receipt* with their own key; `execute()` checks it against the pinned `AGENT_PUB`, so it's refused.
6. **Wrong-action agent receipt** — a validly signed agent receipt for a *different* action is bound onto this execution; refused because the receipt's action must equal the action being executed.
7. **Malformed input** — a structurally broken receipt fails *closed* with a clean reason instead of crashing the verifier. This covers **every** attacker-shaped input: missing fields, bad base64, a **wrong-length signature** (valid base64 that isn't 64 bytes — pynacl raises a `ValueError` that subclasses `CryptoError`, *not* `BadSignatureError`), and a **non-dict payload** (a list where an object is expected, which would otherwise raise `AttributeError` on the field reads). Fail-closed means *no* structurally malformed receipt can ever raise — it always returns `(False, reason)`.

In [6]:
# 1. tamper: change the amount after approval
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_human_approval(approval, APPROVER_PUB, tampered))

# 2. confused deputy: valid approval for action A, used for action B
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_human_approval(approval, APPROVER_PUB, action_b))

# 3. replay: reuse the same approval for a second execution
print("replay              ->", execute(action, approval, agent_receipt(action, approval, 1_750_000_001_000)))

# 4. forgery (human side): attacker signs an APPROVAL with their own key,
#    checked against the pinned approver key
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", 1_750_000_002_000, sk=mallory_sk)
print("forged-approval     ->", verify_human_approval(forged, APPROVER_PUB, action))

# --- the agent-receipt half must be just as unforgeable ---
# A fresh, un-consumed approval so these fail on the agent-receipt check, not on replay.
fresh_approval = human_approval(action, "alice@ops (WebAuthn)", 1_750_000_003_000)

# 5. self-minted agent receipt: attacker signs an AGENT RECEIPT with their own key.
#    execute() checks it against the pinned AGENT_PUB, so the attacker's key is rejected.
mallory_agent_sk = SigningKey.generate()
self_minted = agent_receipt(action, fresh_approval, 1_750_000_003_100, sk=mallory_agent_sk)
print("self-minted-agent   ->", execute(action, fresh_approval, self_minted))

# 6. wrong-action agent receipt: a validly signed (real agent key) receipt for a
#    DIFFERENT action, bound onto this approval; refused because the receipt's action
#    must equal the action being executed.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
wrong_action_rcpt = agent_receipt(wrong_action, fresh_approval, 1_750_000_003_200)
print("wrong-action-agent  ->", execute(action, fresh_approval, wrong_action_rcpt))

# 7. malformed input: structurally broken receipts fail CLOSED, they never crash.
print("malformed-approval  ->", verify_human_approval({"payload": {}}, APPROVER_PUB, action))
print("malformed-agent     ->", execute(action, fresh_approval, {"nope": "not a receipt"}))

# 8. wrong-length signature: valid base64 but NOT 64 bytes ("AAAA" -> 3 bytes). pynacl
#    raises a ValueError (subclass of CryptoError, NOT BadSignatureError) — must still refuse.
badlen_approval = {**human_approval(action, "alice@ops (WebAuthn)", 1_750_000_004_000), "sig": "AAAA"}
print("wrong-len-sig-appr  ->", verify_human_approval(badlen_approval, APPROVER_PUB, action))
badlen_agent = {**agent_receipt(action, fresh_approval, 1_750_000_004_100), "sig": "AAAA"}
print("wrong-len-sig-agent ->", execute(action, fresh_approval, badlen_agent))

# 9. non-dict payload: an attacker sends `payload` as a list, not an object. The .get()
#    calls would raise AttributeError — the isinstance guard makes it a clean refusal.
print("nondict-payload-appr->", verify_human_approval({"payload": [1, 2], "sig": "AAAA"}, APPROVER_PUB, action))
print("nondict-payload-agnt->", execute(action, fresh_approval, {"payload": [1, 2], "sig": "AAAA"}))

tamper              -> (False, 'approval is not for the action being executed (confused deputy)')
confused-deputy     -> (False, 'approval is not for the action being executed (confused deputy)')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'signature invalid (forged or tampered)')
self-minted-agent   -> (False, 'agent receipt signature invalid')
wrong-action-agent  -> (False, 'agent receipt is for a different action than the one being executed')
malformed-approval  -> (False, 'receipt malformed (missing or unreadable fields)')
malformed-agent     -> (False, 'agent receipt malformed')
wrong-len-sig-appr  -> (False, 'signature invalid (forged or tampered)')
wrong-len-sig-agent -> (False, 'agent receipt signature invalid')
nondict-payload-appr-> (False, 'receipt malformed (payload is not an object)')
nondict-payload-agnt-> (False, 'agent receipt malformed')


## What this proves — and what it doesn't

**Proves:** a specific, pinned human key approved *this exact action*; the approval can't be moved to another action, replayed, or forged; and anyone can check it offline without trusting the agent runtime or any server.

**Does not prove:** that the human understood what they approved (that's a UI / WYSIWYS concern — show the human the canonical action), or that the policy was actually enforced (the receipt records what was *claimed*). And the strength of "a human said yes" is only as strong as the authenticator: a real WebAuthn platform authenticator with user-verification is the property that makes it auditable — record *how* the human signed so a verifier can require a device-bound, user-verified approval for high-risk actions.

## Production references

- **WebAuthn / FIDO2** for the human-side signature, so the approval is rooted in a platform authenticator's attestation rather than a software key an operator controls.
- The receipt format and offline verifier in this lesson generalize; the **[EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol)** publishes this human-authorization-receipt shape as IETF Internet-Drafts with an Apache-2.0 cross-language verifier (JavaScript / Python / Go) over the same RFC 8785 canonical bytes, including quorum (multi-approver) and revocation.

## Knowledge check

1. Why must the verifier use the approver's *pinned* public key instead of the one inside the receipt?
2. Why does the human sign the **full canonical action** rather than just an `action_digest`?
3. What stops a valid approval for a $4,200 refund from authorizing a $9,900 one?
